# Hierarchical Data, the JSON Data Format, and APIs

In [1]:
import pandas as pd

## Shows Data

First we'll work with the "Girls" shows JSON data from the reading.

In [2]:
# Fetch data from a URL
import requests
response = requests.get("https://dlsun.github.io/pods/data/tvshows.json")

import json
data_shows = response.json()

In [3]:
df_shows = pd.json_normalize(data_shows)
df_shows

,id,url,name,type,language,genres,status,runtime,premiered,officialSite,...,externals.thetvdb,externals.imdb,image.medium,image.original,network,webChannel.id,webChannel.name,webChannel.country.name,webChannel.country.code,webChannel.country.timezone
0,139,http://www.tvmaze.com/shows/139/girls,Girls,Scripted,English,"[Drama, Romance]",Ended,30,2012-04-15,http://www.hbo.com/girls,...,220411,tt1723816,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
1,722,http://www.tvmaze.com/shows/722/the-golden-girls,The Golden Girls,Scripted,English,"[Drama, Comedy]",Ended,30,1985-09-14,NaN,...,71292,tt0088526,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
2,23542,http://www.tvmaze.com/shows/23542/good-girls,Good Girls,Scripted,English,"[Drama, Comedy, Crime]",Running,60,2018-02-26,https://www.nbc.com/good-girls?nbc=1,...,328577,tt6474378,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
3,6771,http://www.tvmaze.com/shows/6771/the-powerpuff...,The Powerpuff Girls,Animation,English,"[Comedy, Action, Science-Fiction]",Running,15,2016-04-04,https://www.cartoonnetwork.com/video/powerpuff...,...,307473,tt4718304,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
4,42726,http://www.tvmaze.com/shows/42726/florida-girls,Florida Girls,Scripted,English,[Comedy],Running,30,2019-07-10,https://poptv.com/floridagirls,...,363682,tt8548870,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
5,32087,http://www.tvmaze.com/shows/32087/chicken-girls,Chicken Girls,Scripted,English,"[Drama, Children, Music]",Running,16,2017-09-05,https://www.youtube.com/playlist?list=PLVewHiZ...,...,339854,NaN,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,274.0,Brat,United States,US,America/New_York
6,33320,http://www.tvmaze.com/shows/33320/derry-girls,Derry Girls,Scripted,English,[Comedy],Running,30,2018-01-04,http://www.channel4.com/programmes/derry-girls,...,338903,tt7120662,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
7,1955,http://www.tvmaze.com/shows/1955/the-powerpuff...,The Powerpuff Girls,Animation,English,"[Action, Children, Crime]",Ended,30,1998-11-18,NaN,...,76200,tt0175058,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
8,1073,http://www.tvmaze.com/shows/1073/bomb-girls,Bomb Girls,Scripted,English,"[Drama, Romance, War]",Ended,60,2012-01-04,NaN,...,254378,tt1955311,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN
9,525,http://www.tvmaze.com/shows/525/gilmore-girls,Gilmore Girls,Scripted,English,"[Drama, Comedy, Romance]",Ended,60,2000-10-05,NaN,...,76568,tt0238784,http://static.tvmaze.com/uploads/images/medium...,http://static.tvmaze.com/uploads/images/origin...,NaN,NaN,NaN,NaN,NaN,NaN


1\. Summarize the networks represented by these shows and the number of these shows that aired on each network.

In [4]:
df_shows.value_counts("network.name")

network.name
NBC                2
Cartoon Network    2
HBO                1
Pop                1
Channel 4          1
Global             1
The CW             1
Name: count, dtype: int64

2\. Find the number of seasons for each show in the data. Do this two ways: one which uses `df_shows`, and another that first flattens `data_shows` to a different data frame.

In [6]:
df_shows[["id", "name"]].assign(
    num_seasons=df_shows["seasons"].str.len()
)

,id,name,num_seasons
0,139,Girls,6
1,722,The Golden Girls,7
2,23542,Good Girls,3
3,6771,The Powerpuff Girls,3
4,42726,Florida Girls,1
5,32087,Chicken Girls,5
6,33320,Derry Girls,2
7,1955,The Powerpuff Girls,6
8,1073,Bomb Girls,2
9,525,Gilmore Girls,8


In [ ]:
df_seasons = pd.json_normalize(data_shows, record_path="seasons", meta=["id","name"],meta_prefix="show.")

df_seasons.groupby(["show.id", "show.name"]).size().reset_index

<bound method Series.reset_index of show.id  show.name          
139      Girls                  6
525      Gilmore Girls          8
722      The Golden Girls       7
1073     Bomb Girls             2
1955     The Powerpuff Girls    6
6771     The Powerpuff Girls    3
23542    Good Girls             3
32087    Chicken Girls          5
33320    Derry Girls            2
42726    Florida Girls          1
dtype: int64>

3\. For each episode, find the length (number of characters) of the title. Then create summaries to answer: Which show tends to have the longest episode titles? The shortest?

In [10]:
df_episodes = pd.json_normalize(
    data_shows,
    record_path=["seasons", "episodes"],
    meta=["id", "name"],
    meta_prefix="show."
)

df_episodes["title_length"] = df_episodes["name"].str.len()

In [12]:
title_summary = (
    df_episodes
    .groupby(["show.id", "show.name"])["title_length"]
    .agg(["count", "mean", "min", "max"])
    .sort_values("mean", ascending=False)
).reset_index()

title_summary

,show.id,show.name,count,mean,min,max
0,1955,The Powerpuff Girls,82,27.951220,9,66
1,525,Gilmore Girls,153,21.431373,5,51
2,42726,Florida Girls,10,20.400000,5,30
3,722,The Golden Girls,181,19.723757,6,53
4,6771,The Powerpuff Girls,119,16.521008,5,42
5,32087,Chicken Girls,76,16.434211,5,70
6,23542,Good Girls,26,15.461538,4,49
7,1073,Bomb Girls,19,14.421053,8,21
8,139,Girls,63,13.111111,3,42
9,33320,Derry Girls,12,12.500000,8,35


4\. Do any cast members in the data set share a birthday with you? Who, and what show are they on? (If there isn't anyone, try a different day.)

In [13]:
df_cast = pd.json_normalize(
    data_shows,
    record_path="cast",
    meta=["id", "name"],
    meta_prefix="show."
)

df_cast[["person.name", "person.birthday", "show.name"]].head()

,person.name,person.birthday,show.name
0,Lena Dunham,1986-05-13,Girls
1,Allison Williams,1988-04-13,Girls
2,Jemima Kirke,1985-04-26,Girls
3,Zosia Mamet,1988-02-02,Girls
4,Adam Driver,1983-11-19,Girls


In [18]:
df_cast[df_cast["person.birthday"].str.endswith("05-22", na=False)][["person.name", "person.birthday", "show.name"]]

,person.name,person.birthday,show.name
103,Sean Gunn,1974-05-22,Gilmore Girls


## TVMaze API

Now you will work with the [TVMaze API](http://www.tvmaze.com/api) from the reading. Use the API to request JSON data that you can use to answer the following questions.

1\. What was the longest show that aired in the U.S. on February 4, 2018?

_Hint:_ Use the ["Schedule" endpoint](http://www.tvmaze.com/api#schedule) to first get the data for all shows that aired on that date.

In [30]:
response = requests.get("http://api.tvmaze.com/schedule",
    params={"country": "US", "date": "2018-02-04"}
)
schedule = response.json()
df_schedule = pd.json_normalize(schedule)
df_schedule[["show.id", "show.name", "show.runtime"]].sort_values("show.runtime", ascending=False).head(1)

,show.id,show.name,show.runtime
23,6011,Super Bowl,210.0


2\. Among all shows that aired in the U.S. on Feburary 4, 2018, which non-voice actors appeared on more than one show? Note: some people are credited with multiple rows on the same show and thus appear as multiple rows in the data frame.

Hint: You will need to write a for loop to make multiple requests. Use the "show.id" from the data from part 1, and use the ["Shows" endpoint](http://www.tvmaze.com/api#show-cast) to get the cast of each show. Don't forget to stagger your requests, or you will be blocked by the website!

In [38]:
import time
show_ids = df_schedule["show.id"].to_list()
show_names = df_schedule.set_index("show.id")["show.name"].to_dict()

cast = []

for show_id in show_ids:
    response = requests.get(f"https://api.tvmaze.com/shows/{show_id}/cast")
    members = response.json()
    for member in members:
        member["show_id"] = show_id
        member["show_name"] = show_names[show_id]
    cast.extend(members)
    time.sleep(0.5)

In [39]:
df_cast = pd.json_normalize(cast)
df_cast.head()

,self,voice,show_id,show_name,person.id,person.url,person.name,person.country.name,person.country.code,person.country.timezone,...,person._links.self.href,character.id,character.url,character.name,character.image.medium,character.image.original,character._links.self.href,person.country,person.image,character.image
0,True,False,7795,1st Look,117627,https://www.tvmaze.com/people/117627/johnny-de...,Johnny Devenanzio,United States,US,America/New_York,...,https://api.tvmaze.com/people/117627,627654,https://www.tvmaze.com/characters/627654/1st-l...,Host,https://static.tvmaze.com/uploads/images/mediu...,https://static.tvmaze.com/uploads/images/origi...,https://api.tvmaze.com/characters/627654,NaN,NaN,NaN
1,True,False,7795,1st Look,132602,https://www.tvmaze.com/people/132602/angela-sun,Angela Sun,NaN,NaN,NaN,...,https://api.tvmaze.com/people/132602,627644,https://www.tvmaze.com/characters/627644/1st-l...,Host,https://static.tvmaze.com/uploads/images/mediu...,https://static.tvmaze.com/uploads/images/origi...,https://api.tvmaze.com/characters/627644,NaN,NaN,NaN
2,True,False,7795,1st Look,71115,https://www.tvmaze.com/people/71115/ashley-rob...,Ashley Roberts,United States,US,America/New_York,...,https://api.tvmaze.com/people/71115,627646,https://www.tvmaze.com/characters/627646/1st-l...,Host,https://static.tvmaze.com/uploads/images/mediu...,https://static.tvmaze.com/uploads/images/origi...,https://api.tvmaze.com/characters/627646,NaN,NaN,NaN
3,True,False,7795,1st Look,51988,https://www.tvmaze.com/people/51988/audrina-pa...,Audrina Patridge,United States,US,America/New_York,...,https://api.tvmaze.com/people/51988,251236,https://www.tvmaze.com/characters/251236/1st-l...,Host,https://static.tvmaze.com/uploads/images/mediu...,https://static.tvmaze.com/uploads/images/origi...,https://api.tvmaze.com/characters/251236,NaN,NaN,NaN
4,True,False,7795,1st Look,235612,https://www.tvmaze.com/people/235612/pedro-and...,Pedro Andrade,NaN,NaN,NaN,...,https://api.tvmaze.com/people/235612,627647,https://www.tvmaze.com/characters/627647/1st-l...,Host,https://static.tvmaze.com/uploads/images/mediu...,https://static.tvmaze.com/uploads/images/origi...,https://api.tvmaze.com/characters/627647,NaN,NaN,NaN


In [41]:
non_voice = df_cast[df_cast["voice"] == False]

non_voice = non_voice.drop_duplicates(subset=["person.id", "show_id"])

actor_count = (
    non_voice.groupby(["person.id", "person.name"])["show_id"].nunique().reset_index(name="number_of_shows")
)

actor_count[actor_count["number_of_shows"] > 1]

,person.id,person.name,number_of_shows
83,102424,John Dickerson,2
133,162299,Tom Llamas,2


# Tasty API

[Tasty.co](http://tasty.co) is a website and app that offers food recipes. They have made these recipes available through a [a REST API](https://rapidapi.com/apidojo/api/tasty). However, unlike the TVMaze API, this one requires authentication.

Specifically, you will need to create an account and subscribe to the "Basic" (free) plant. You will then be provided with an API key (X-Rapid-API-Key) that will need to be supplied with every request you make. This is used to track and limit usage.

1. [create an account](https://rapidapi.com/apidojo/api/tasty) and subscribe to the "Basic" (free) plan
2. log in and copy the X-RapidAPI-Key, which is a long string of letters and digits
3. paste this key to replace "PUT-YOUR-KEY-HERE" in the `headers` below

If you did everything correctly, then running the cell below should return a JSON object containing all the tags recognized by the Tasty API.

In [42]:
import requests

domain = "https://tasty.p.rapidapi.com"
endpoint = "tags/list"
url = f"{domain}/{endpoint}"

# TODO: Update the `headers` with your X-RapidAPI-Key.
headers = {
	"X-RapidAPI-Key": "2c42d54a16msh96a8e46aca45e95p1dd1efjsn04f727807c0b",
	"X-RapidAPI-Host": "tasty.p.rapidapi.com"
}

# Make an HTTP request to the REST API, get the JSON response.
response = requests.get(url, headers=headers)
response.json()

{'count': 588,
 'results': [{'root_tag_type': 'cuisine',
   'name': 'brazilian',
   'id': 64446,
   'parent_tag_name': 'central_south_american',
   'display_name': 'Brazilian',
   'type': 'central_south_american'},
  {'root_tag_type': 'cuisine',
   'name': 'german',
   'id': 64450,
   'parent_tag_name': 'european',
   'display_name': 'German',
   'type': 'european'},
  {'root_tag_type': 'healthy',
   'name': 'healthy',
   'id': 64466,
   'parent_tag_name': None,
   'display_name': 'Healthy',
   'type': 'healthy'},
  {'root_tag_type': 'dietary',
   'name': 'vegetarian',
   'id': 64469,
   'parent_tag_name': 'dietary',
   'display_name': 'Vegetarian',
   'type': 'dietary'},
  {'root_tag_type': 'seasonal',
   'name': 'christmas',
   'id': 64473,
   'parent_tag_name': 'holidays',
   'display_name': 'Christmas',
   'type': 'holidays'},
  {'root_tag_type': 'seasonal',
   'name': 'halloween',
   'id': 64476,
   'parent_tag_name': 'holidays',
   'display_name': 'Halloween',
   'type': 'holiday

You will need to pass these `headers` with every HTTP request to the API. The API key (X-RapidAPI-Key) is how the server keeps track of how many requests you have made.

Take a look at [the documentation](https://rapidapi.com/apidojo/api/tasty). We will use the recipes/list endpoint.

Make sure you are logged into the account you are created, and select the recipes/list endpoint from the menu at left. Notice that this brings up a form that you can fill in, which generates the corresponding code.  By default, it provides Node.js code; change this to Python with the Requests client.

1\. Search for recipes containing "daikon" (an Asian radish) and request the JSON data, and convert it to a Pandas data frame.

In [57]:
import requests

url = "https://tasty.p.rapidapi.com/recipes/list"

querystring = {"q":"daikon","size":"50","from":"0"}

headers = {
	"x-rapidapi-key": "2c42d54a16msh96a8e46aca45e95p1dd1efjsn04f727807c0b",
	"x-rapidapi-host": "tasty.p.rapidapi.com"
}

response = requests.get(url, headers=headers, params=querystring)
response = response.json()

daikon = pd.json_normalize(response)

2\. How many recipes containing daikon are there? Which one is the cheapest per portion?

In [58]:
daikon.loc[0, "count"]

np.int64(16)

In [59]:
df_daikon = pd.json_normalize(daikon.loc[0, "results"])
df_daikon.columns[df_daikon.columns.str.contains("price", case=False)]
df_daikon.loc[
    df_daikon["price.portion"].idxmin(),
    ["name", "price.portion"]
]

name             Vegan Tofu Bao Buns With Pickled Vegetables
price.portion                                          300.0
Name: 10, dtype: object

3\. Find recipes containing avocado and request the JSON data.

**Hint:** Note that there are hundreds of results, but the API only returns 20 results by default and only 40 results maximum, even if you specify the `size=` parameter, so you will need to use a `for` loop, incrementing the `from=` parameter. Be sure to respect the API's rate limits (or you may be blocked!)

**Note:** Recall from the example in the reading that we created an empty list `episodes = []` and then added the results of each request to it in the for loop with `episodes.extend(response.json())`. But note that in the recipes/list endpoint there are two keys: count and results. We only want the results so try instead `.extend(response.json()).get("results")`.

**Suggestion:** Try writing a loop that only makes 2 or 3 requests first so you can test that it's working correctly. Also, make sure you use separate cells for code that makes requests to the API versus processing of the results; you don't want to rerun the requests unless absolutely necessary!

In [61]:

avocado_recipes = []

# First request
response = requests.get(
    url,
    headers=headers,
    params={"q": "avocado", "from": 0, "size": 40}
)

data = response.json()
total_recipes = data["count"]
avocado_recipes.extend(data.get("results", []))

time.sleep(0.5)

# Remaining requests
for start in range(40, total_recipes, 40):
    response = requests.get(
        url,
        headers=headers,
        params={"q": "avocado", "from": start, "size": 40}
    )

    batch = response.json().get("results", [])
    avocado_recipes.extend(batch)

    time.sleep(0.5)

df_avocado = pd.json_normalize(avocado_recipes)

df_avocado.head()

,nutrition_visibility,country,instructions,keywords,facebook_posts,language,seo_path,id,brand,slug,...,show.name,show.id,total_time_tier.tier,total_time_tier.display_tier,total_time_tier,brand.image_url,brand.name,brand.id,brand.slug,recipes
0,auto,US,"[{'start_time': 0, 'appliance': None, 'end_tim...",", avocado, buzzfeedtasty, carbonara, dinner, e...",[],eng,"8757513,9295874,64453",56,NaN,avocado-carbonara,...,Tasty,17,under_30_minutes,Under 30 minutes,NaN,NaN,NaN,NaN,NaN,NaN
1,auto,ZZ,"[{'start_time': 0, 'appliance': 'oven', 'end_t...",NaN,[],eng,"9295813,64486,64459",1340,NaN,avocado-lime-salmon,...,Tasty,17,under_30_minutes,Under 30 minutes,NaN,NaN,NaN,NaN,NaN,NaN
2,auto,US,"[{'start_time': 4833, 'appliance': None, 'hack...","avocado, cucumber, easy, quick, quinoa, salad,...",[],eng,"9295813,64489,9299495",3932,NaN,avocado-quinoa-power-salad,...,Tasty: Tasty Vegetarian,49,under_1_hour,Under 1 hour,NaN,NaN,NaN,NaN,NaN,NaN
3,auto,US,"[{'start_time': 0, 'appliance': 'oven', 'end_t...","avocado, ceviche, cilantro, citrus, cocktail, ...",[],eng,"8757513,64444,64457",2819,NaN,shrimp-avocado-tostadas,...,Goodful,34,under_45_minutes,Under 45 minutes,NaN,NaN,NaN,NaN,NaN,NaN
4,auto,US,"[{'start_time': 0, 'appliance': None, 'hacks':...",NaN,[],eng,"9295816,8091744",3363,NaN,avocado-toast,...,Tasty: Tasty Vegetarian,49,under_15_minutes,Under 15 minutes,NaN,NaN,NaN,NaN,NaN,NaN


4\. For the recipes containing avocado, compute the proportion of reviews that are positive. For the avocado recipes with over 500 reviews, which one has the highest proportion of positive reviews?


In [65]:
df_avocado["total_reviews"] = (
    df_avocado["user_ratings.count_positive"]
    + df_avocado["user_ratings.count_negative"]
)

df_avocado["positive_proportion"] = (
    df_avocado["user_ratings.count_positive"]
    / df_avocado["total_reviews"]
)

over_500 = df_avocado[df_avocado["total_reviews"] > 500]

over_500.loc[
    over_500["positive_proportion"].idxmax(),
    ["name", "total_reviews", "positive_proportion"]
]

name                   Grilled Salmon With Avocado Salsa
total_reviews                                     1898.0
positive_proportion                             0.987355
Name: 12, dtype: object

5\. Take the avocado JSON data from above (you do NOT need to read in the data from the REST API again). How many recipes are vegetarian? You should be able to identify this from the "tags" attribute.

Hint: Try using `json_normalize` with "tags" as the record path to flatten the data so that there is one row for each tag.

In [66]:
df_tags = pd.json_normalize(
    avocado_recipes,
    record_path="tags",
    meta=["id", "name"],
    meta_prefix="recipe."
)

vegetarian_tags = df_tags[
    df_tags["name"].str.lower().eq("vegetarian")
]

vegetarian_tags["recipe.id"].nunique()

209